# Causal Inference for HR Interventions
## Estimating the True Impact of Manager Training on Team Attrition

**Author:** [Sunidhi Sharma](https://www.linkedin.com/in/sunidhisharma28/) | Senior Data Scientist, People Analytics & Workforce Intelligence

---

### Project Overview

**Objective:** Estimate the causal effect of a manager training program on team-level attrition using quasi-experimental methods, and translate those estimates into actionable ROI calculations for HR leadership.

**Core Question:** HR invested in a leadership development program. Managers who attended now have lower team attrition. But did the training *cause* that improvement, or were better managers simply more likely to be selected for the program in the first place?

**Approach:** Apply two complementary causal inference methods (Propensity Score Matching and Difference-in-Differences) to isolate the true treatment effect from selection bias, then assess robustness through sensitivity analysis and explore heterogeneous treatment effects across departments.

**Key Deliverables:**

| Deliverable | What It Provides | Who Uses It |
|---|---|---|
| Selection bias diagnosis | Evidence that naive comparisons are unreliable | Analytics leads, HR leadership |
| Propensity Score Matching estimate | Causal effect under conditional independence assumption | HR Strategy, L&D |
| Difference-in-Differences estimate | Causal effect under parallel trends assumption | HR Strategy, L&D |
| Sensitivity analysis | How robust the estimate is to unmeasured confounders | Analytics leads, Legal |
| Heterogeneous treatment effects | Which departments benefit most from the training | L&D, Workforce Planning |
| ROI calculation | Dollar value per manager trained, with targeting recommendations | CHRO, CFO |

**Tech Stack:** Python, pandas, scikit-learn, statsmodels, scipy, matplotlib, seaborn

**Note:** This project uses simulated data with a known true effect, allowing us to validate that our causal methods recover the correct answer. In production, the same methods would be applied to real HRIS data.

---

### Why Causal Inference? A Methodological Note

Most People Analytics teams operate in one of two modes:

1. **Descriptive analytics:** "Managers who attended training have 8% lower team attrition."
2. **Predictive analytics:** "This model predicts which managers' teams are at high attrition risk."

Neither of these answers the question leadership actually needs: *"Did the training program cause the improvement, and should we invest more in it?"*

That question requires **causal inference**, which is fundamentally different from prediction or description.

| Approach | Question It Answers | Limitation |
|---|---|---|
| Descriptive (group means) | What happened? | Cannot separate correlation from causation |
| Predictive (ML models) | What will happen? | Optimized for prediction, not for understanding *why* |
| Causal (this project) | What would happen if we intervene? | Requires careful assumptions, but produces decision-grade evidence |

In my previous work at Publicis Sapient, I applied A/B testing and difference-in-differences to evaluate sourcing strategies on Workday ATS data, identifying decision frictions that led to policy reforms reducing time-to-fill by 25%. This project extends those methods to a different HR intervention: manager training.

The two methods used here are:

**Propensity Score Matching (PSM):** Creates a pseudo-randomized comparison by matching treated managers to similar untreated managers based on observable characteristics. Relies on the **Conditional Independence Assumption** (no unmeasured confounders).

**Difference-in-Differences (DiD):** Compares the pre-to-post change in treated vs control groups. Relies on the **Parallel Trends Assumption** (absent treatment, both groups would have followed the same trajectory).

Using both methods provides a robustness check: if PSM and DiD produce similar estimates, our confidence in the causal claim is substantially stronger.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Professional styling
sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'grid.alpha': 0.3
})

COLORS = {
    'treated': '#E74C3C',
    'control': '#3498DB',
    'accent': '#2ECC71',
    'neutral': '#7F8C8D',
    'dark': '#2C3E50'
}

np.random.seed(42)
%matplotlib inline

## 1. Data Generation & Selection Bias Diagnosis

We simulate a realistic HR dataset because real organizational data is proprietary. Critically, the simulation encodes **selection bias**: better managers (higher ratings, more experience) are more likely to be enrolled in the training program. This is exactly what happens in practice, because HR teams tend to invest in high-potential managers or nominate managers from high-priority teams.

The simulation also encodes a **known true treatment effect** of -4 percentage points. This ground truth lets us validate that our causal methods actually recover the correct answer, something that is impossible with real observational data.

In [ ]:
def simulate_manager_training_data(n=500, true_ate=-0.04, seed=42):
    """
    Simulate manager-level data with a training intervention.
    
    Key design: treatment assignment is NOT random.
    Better managers and larger teams are more likely to be selected,
    creating the selection bias that makes causal inference necessary.
    """
    np.random.seed(seed)
    
    # Manager characteristics (confounders)
    manager_experience = np.random.gamma(shape=5, scale=2, size=n)
    team_size = np.random.poisson(lam=12, size=n).clip(3, 40)
    department = np.random.choice(['Engineering', 'Sales', 'Operations', 'Support'], 
                                   size=n, p=[0.3, 0.25, 0.25, 0.2])
    manager_rating = np.random.normal(3.5, 0.7, size=n).clip(1, 5)
    
    # Department-level attrition baselines
    dept_baseline = {'Engineering': 0.12, 'Sales': 0.18, 'Operations': 0.10, 'Support': 0.22}
    base_attrition = np.array([dept_baseline[d] for d in department])
    
    # Pre-treatment attrition (function of confounders + noise)
    pre_attrition = (
        base_attrition
        - 0.008 * manager_experience
        - 0.015 * (manager_rating - 3.5)
        + 0.001 * team_size
        + np.random.normal(0, 0.03, size=n)
    ).clip(0.02, 0.40)
    
    # Treatment assignment: biased toward better managers (SELECTION BIAS)
    propensity = (
        -1.0
        + 0.08 * manager_experience
        + 0.4 * manager_rating
        + 0.02 * team_size
        + np.where(department == 'Engineering', 0.3, 0)
        + np.random.normal(0, 0.5, size=n)
    )
    treatment_prob = 1 / (1 + np.exp(-propensity))
    treatment = np.random.binomial(1, treatment_prob)
    
    # Post-treatment attrition with heterogeneous treatment effects
    time_trend = -0.01  # natural improvement for everyone
    hte = np.where(department == 'Support', true_ate * 1.5,
           np.where(department == 'Sales', true_ate * 1.2,
                    true_ate))
    
    post_attrition = (
        pre_attrition + time_trend + treatment * hte
        + np.random.normal(0, 0.025, size=n)
    ).clip(0.01, 0.40)
    
    return pd.DataFrame({
        'manager_id': range(1, n + 1),
        'manager_experience': manager_experience.round(1),
        'team_size': team_size,
        'department': department,
        'manager_rating': manager_rating.round(2),
        'pre_attrition_rate': pre_attrition.round(4),
        'post_attrition_rate': post_attrition.round(4),
        'treatment': treatment,
        'treatment_prob': treatment_prob.round(4)
    }), true_ate

df, TRUE_ATE = simulate_manager_training_data(n=500, true_ate=-0.04)

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Total managers:     {len(df)}")
print(f"Treated (trained):  {df['treatment'].sum()} ({df['treatment'].mean():.1%})")
print(f"Control:            {(1 - df['treatment']).sum():.0f} ({1 - df['treatment'].mean():.1%})")
print(f"\nTrue ATE (ground truth): {TRUE_ATE:.1%} reduction in attrition rate")
print(f"\nDepartment distribution:")
for dept in ['Engineering', 'Sales', 'Operations', 'Support']:
    n_dept = (df['department'] == dept).sum()
    treat_rate = df[df['department'] == dept]['treatment'].mean()
    print(f"  {dept:15s}  n={n_dept:3d}  treatment rate: {treat_rate:.0%}")
df.head()

In [ ]:
# Quantify the selection bias
print("=" * 65)
print("EVIDENCE OF SELECTION BIAS")
print("=" * 65)
comparison = df.groupby('treatment').agg({
    'manager_experience': 'mean',
    'team_size': 'mean',
    'manager_rating': 'mean',
    'pre_attrition_rate': 'mean',
    'post_attrition_rate': 'mean'
}).round(3)
comparison.index = ['Control', 'Treated']
print(comparison.to_string())

naive = comparison.loc['Treated', 'post_attrition_rate'] - comparison.loc['Control', 'post_attrition_rate']
print(f"\nNaive estimate (post mean difference): {naive:.4f} ({naive:.1%})")
print(f"True ATE:                              {TRUE_ATE:.4f} ({TRUE_ATE:.1%})")
print(f"Bias (naive - true):                   {naive - TRUE_ATE:+.4f}")
print(f"\nThe naive estimate OVERSTATES the effect by {abs(naive)/abs(TRUE_ATE):.1f}x")
print(f"because treated managers were already better before the program.")

In [ ]:
# Visualize the selection bias
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for label, color in [(0, COLORS['control']), (1, COLORS['treated'])]:
    name = 'Treated' if label else 'Control'
    subset = df[df['treatment'] == label]
    axes[0].hist(subset['manager_experience'], bins=20, alpha=0.6, color=color, label=name, density=True)
    axes[1].hist(subset['manager_rating'], bins=20, alpha=0.6, color=color, label=name, density=True)
    axes[2].hist(subset['pre_attrition_rate'], bins=20, alpha=0.6, color=color, label=name, density=True)

axes[0].set_title('Manager Experience by Group', fontweight='bold')
axes[0].set_xlabel('Years of Experience')
axes[1].set_title('Manager Rating by Group', fontweight='bold')
axes[1].set_xlabel('Rating (1-5)')
axes[2].set_title('Pre-Treatment Attrition by Group', fontweight='bold')
axes[2].set_xlabel('Attrition Rate')

for ax in axes:
    ax.legend()

plt.suptitle('Selection Bias: Treated Managers Differ Systematically from Control',
             fontsize=14, fontweight='bold', y=1.03, color=COLORS['dark'])
plt.tight_layout()
plt.savefig('../outputs/figures/selection_bias_evidence.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretation: Selection Bias

The three histograms reveal the core problem: treated and control groups are **not comparable before the intervention even starts.**

Treated managers have higher experience, higher ratings, and lower pre-treatment attrition. This means any post-treatment difference in attrition conflates two effects: (1) the genuine training impact and (2) the fact that better managers were selected. The naive estimate absorbs both, which is why it overstates the effect by roughly 2x.

This is not a hypothetical concern. In real organizations, training programs are almost never randomly assigned. HR teams nominate high-potential managers, or managers self-select based on motivation. Without causal methods, every program evaluation is contaminated by this bias.

## 2. Propensity Score Matching

**The idea:** Estimate each manager's probability of receiving treatment (the propensity score), then match each treated manager to a control manager with the most similar propensity score. This creates a pseudo-randomized comparison that adjusts for observed confounders.

**Key assumption (Conditional Independence):** Given the observed covariates, treatment assignment is independent of potential outcomes. In simpler terms: we have measured everything that determines both who gets trained and what their attrition would be.

### Step 1: Estimate Propensity Scores

In [ ]:
# Propensity score model: P(treatment | covariates)
covariates = ['manager_experience', 'team_size', 'manager_rating', 'pre_attrition_rate']
dept_dummies = pd.get_dummies(df['department'], prefix='dept', drop_first=True)

X_ps = pd.concat([df[covariates], dept_dummies], axis=1)
y_ps = df['treatment']

ps_model = LogisticRegression(max_iter=1000, random_state=42)
ps_model.fit(X_ps, y_ps)
df['propensity_score'] = ps_model.predict_proba(X_ps)[:, 1]

# Check overlap (positivity assumption)
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df.loc[df['treatment'] == 0, 'propensity_score'], bins=30, alpha=0.6, 
        color=COLORS['control'], label='Control', density=True)
ax.hist(df.loc[df['treatment'] == 1, 'propensity_score'], bins=30, alpha=0.6,
        color=COLORS['treated'], label='Treated', density=True)
ax.set_xlabel('Propensity Score', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Propensity Score Distributions: Checking Overlap (Positivity)', fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../outputs/figures/propensity_overlap.png', dpi=150, bbox_inches='tight')
plt.show()

print("Propensity score summary by group:")
print(df.groupby('treatment')['propensity_score'].describe().round(3).to_string())

### Interpretation: Propensity Score Overlap

The **positivity (overlap) assumption** requires that both treated and control managers exist across the full range of propensity scores. If there were regions where only treated managers exist, we could not construct valid matches there.

The histogram shows good overlap in the middle range but some separation in the tails. This is typical for observational HR data and is exactly why we apply a caliper (maximum allowed distance) during matching: we discard treated managers who have no comparable control, rather than forcing bad matches.

### Step 2: Nearest-Neighbor Matching with Caliper

In [ ]:
# 1:1 nearest neighbor matching on propensity score
treated = df[df['treatment'] == 1].copy()
control = df[df['treatment'] == 0].copy()

nn = NearestNeighbors(n_neighbors=1, metric='euclidean')
nn.fit(control[['propensity_score']])
distances, indices = nn.kneighbors(treated[['propensity_score']])

matched_control_idx = control.index[indices.flatten()]
matched_control = control.loc[matched_control_idx].copy()
matched_treated = treated.copy()

# Apply caliper: drop matches with propensity score distance > 0.05
caliper = 0.05
good_matches = distances.flatten() < caliper
matched_treated = matched_treated.iloc[good_matches]
matched_control = matched_control.iloc[good_matches]

print(f"Matching results:")
print(f"  Total treated:    {len(treated)}")
print(f"  Matched pairs:    {len(matched_treated)}")
print(f"  Dropped (caliper): {(~good_matches).sum()}")
print(f"  Caliper used:     {caliper}")

### Step 3: Covariate Balance Check

After matching, we verify that the treated and control groups are now **balanced** on observable characteristics. The standard diagnostic is the **Standardized Mean Difference (SMD)**: values below 0.1 indicate acceptable balance.

In [ ]:
def standardized_mean_diff(treated_df, control_df, col):
    """Compute standardized mean difference for balance check."""
    d = treated_df[col].mean() - control_df[col].mean()
    s = np.sqrt((treated_df[col].var() + control_df[col].var()) / 2)
    return d / s if s > 0 else 0

balance_cols = covariates
smd_before, smd_after = [], []

print("=" * 65)
print("COVARIATE BALANCE CHECK (|SMD| < 0.1 is acceptable)")
print("=" * 65)
print(f"{'Covariate':<25} {'Before':>10} {'After':>10} {'Status':>10}")
print("-" * 60)

for col in balance_cols:
    before = standardized_mean_diff(treated, control, col)
    after = standardized_mean_diff(matched_treated, matched_control, col)
    smd_before.append(abs(before))
    smd_after.append(abs(after))
    status = 'PASS' if abs(after) < 0.1 else 'CHECK'
    print(f"{col:<25} {before:>10.3f} {after:>10.3f} {status:>10}")

# Love plot
fig, ax = plt.subplots(figsize=(8, 5))
y_pos = range(len(balance_cols))
ax.scatter(smd_before, y_pos, marker='x', s=100, color=COLORS['treated'], label='Before matching', zorder=3)
ax.scatter(smd_after, y_pos, marker='o', s=100, color=COLORS['accent'], label='After matching', zorder=3)
ax.axvline(x=0.1, color=COLORS['neutral'], linestyle='--', alpha=0.5, label='|SMD| = 0.1 threshold')
ax.set_yticks(list(y_pos))
ax.set_yticklabels(balance_cols)
ax.set_xlabel('|Standardized Mean Difference|')
ax.set_title('Love Plot: Covariate Balance Before/After Matching', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../outputs/figures/love_plot_balance.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretation: Covariate Balance (Love Plot)

The Love Plot is the standard diagnostic in causal inference for assessing matching quality. Each row represents one covariate:

**Before matching (red X):** Large imbalances, especially on manager experience and rating. This confirms the selection bias we diagnosed earlier.

**After matching (green dots):** All covariates fall below the 0.1 SMD threshold. This means that within our matched sample, treated and control managers are now comparable on observable characteristics. The matching has successfully constructed a valid counterfactual.

This balance check is not optional. Without it, the PSM estimate is no more credible than the naive comparison. It is the matching equivalent of checking model assumptions.

### Step 4: Estimate Causal Effect from Matched Sample

In [ ]:
# Average Treatment Effect on the Treated (ATT)
att_psm = matched_treated['post_attrition_rate'].mean() - matched_control['post_attrition_rate'].mean()

# Bootstrap confidence interval
n_bootstrap = 2000
bootstrap_atts = []
for _ in range(n_bootstrap):
    idx = np.random.choice(len(matched_treated), size=len(matched_treated), replace=True)
    bt = matched_treated.iloc[idx]['post_attrition_rate'].values - matched_control.iloc[idx]['post_attrition_rate'].values
    bootstrap_atts.append(bt.mean())

ci_lower = np.percentile(bootstrap_atts, 2.5)
ci_upper = np.percentile(bootstrap_atts, 97.5)
se = np.std(bootstrap_atts)

print("=" * 60)
print("PROPENSITY SCORE MATCHING: CAUSAL ESTIMATE")
print("=" * 60)
print(f"Estimated ATT:    {att_psm:.4f} ({att_psm:.1%})")
print(f"95% CI (boot):    [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"Standard Error:   {se:.4f}")
print(f"p-value (approx): {2 * (1 - stats.norm.cdf(abs(att_psm / se))):.4f}")
print(f"\nTrue ATE:         {TRUE_ATE:.4f}")
print(f"Bias:             {att_psm - TRUE_ATE:+.4f}")
print(f"CI contains true? {'Yes' if ci_lower <= TRUE_ATE <= ci_upper else 'No'}")

### Interpretation: PSM Result

The PSM estimate is substantially closer to the true effect than the naive comparison. Where the naive comparison suggested roughly an 8pp effect (inflated by selection bias), the PSM estimate recovers something much closer to the true 4pp.

The 95% bootstrap confidence interval contains the true effect, indicating the method is properly calibrated. The remaining bias (the small gap between estimate and truth) is expected because PSM can only adjust for *observed* confounders. Any unmeasured factors (e.g., manager motivation, which is hard to quantify) could still introduce residual bias.

This is why we also run a Difference-in-Differences estimate and a sensitivity analysis, as a triangulation strategy.

## 3. Difference-in-Differences

DiD takes a fundamentally different approach: instead of matching on characteristics, it uses each manager as their own control by comparing their **change** in attrition from pre to post. The causal effect is the difference in changes between treated and control groups.

**Key assumption (Parallel Trends):** In the absence of treatment, treated and control groups would have followed the same trend. This is untestable but can be assessed by checking pre-treatment trends when multiple pre-periods are available.

The DiD model: `attrition = beta_0 + beta_1 * treatment + beta_2 * post + beta_3 * (treatment x post) + error`

The coefficient **beta_3** (the interaction term) is the causal estimate.

In [ ]:
# Reshape to panel format
df_pre = df[['manager_id', 'treatment', 'pre_attrition_rate']].copy()
df_pre['period'] = 0
df_pre = df_pre.rename(columns={'pre_attrition_rate': 'attrition_rate'})

df_post = df[['manager_id', 'treatment', 'post_attrition_rate']].copy()
df_post['period'] = 1
df_post = df_post.rename(columns={'post_attrition_rate': 'attrition_rate'})

panel = pd.concat([df_pre, df_post], ignore_index=True)
panel['treated_x_post'] = panel['treatment'] * panel['period']

# DiD regression with clustered standard errors (by manager)
did_model = smf.ols('attrition_rate ~ treatment + period + treated_x_post', data=panel).fit(
    cov_type='cluster', cov_kwds={'groups': panel['manager_id']}
)

print("=" * 60)
print("DIFFERENCE-IN-DIFFERENCES REGRESSION")
print("=" * 60)
print(did_model.summary().tables[1])

did_effect = did_model.params['treated_x_post']
did_ci = did_model.conf_int().loc['treated_x_post']
did_pvalue = did_model.pvalues['treated_x_post']

print(f"\nDiD Causal Estimate: {did_effect:.4f} ({did_effect:.1%})")
print(f"95% CI:              [{did_ci[0]:.4f}, {did_ci[1]:.4f}]")
print(f"p-value:             {did_pvalue:.4f}")
print(f"True ATE:            {TRUE_ATE:.4f}")
print(f"Bias:                {did_effect - TRUE_ATE:+.4f}")

In [ ]:
# DiD visualization
fig, ax = plt.subplots(figsize=(10, 6))

means = panel.groupby(['period', 'treatment'])['attrition_rate'].mean().unstack()

ax.plot([0, 1], means[0], 'o-', color=COLORS['control'], linewidth=2.5, markersize=10, label='Control')
ax.plot([0, 1], means[1], 's-', color=COLORS['treated'], linewidth=2.5, markersize=10, label='Treated')

# Counterfactual line (what would have happened without treatment)
counterfactual_post = means[1][0] + (means[0][1] - means[0][0])
ax.plot([0, 1], [means[1][0], counterfactual_post], 's--', color=COLORS['treated'], 
        alpha=0.4, markersize=8, label='Treated (counterfactual)')

# Annotate the DiD effect
ax.annotate('', xy=(1.05, means[1][1]), xytext=(1.05, counterfactual_post),
            arrowprops=dict(arrowstyle='<->', color='black', lw=2))
ax.text(1.1, (means[1][1] + counterfactual_post) / 2, f'DiD = {did_effect:.3f}',
        fontsize=12, fontweight='bold', va='center')

ax.set_xticks([0, 1])
ax.set_xticklabels(['Pre-Training', 'Post-Training'], fontsize=12)
ax.set_ylabel('Attrition Rate', fontsize=12)
ax.set_title('Difference-in-Differences: Manager Training Impact', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../outputs/figures/did_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretation: Difference-in-Differences

The DiD plot is the most intuitive visualization of the causal logic:

**Solid lines** show what actually happened: both groups improved over time, but the treated group improved *more*.

**Dashed line** shows the counterfactual: what would have happened to the treated group if they had followed the same trend as control. The gap between actual treated outcome and this counterfactual is the causal estimate.

The DiD estimate is close to the PSM estimate, which is reassuring. When two methods with different assumptions (conditional independence for PSM, parallel trends for DiD) converge on similar answers, our confidence in the causal claim is substantially stronger.

The **clustered standard errors** (by manager) account for the fact that each manager contributes two observations (pre and post), which are not independent. Without clustering, standard errors would be artificially small and p-values misleadingly low.

## 4. Heterogeneous Treatment Effects

A single average treatment effect is useful for the overall go/no-go decision, but HR leadership also needs to know: **where should we target future cohorts?** If the training works twice as well in Support as in Engineering, the budget allocation should reflect that.

In [ ]:
# DiD by department
df['attrition_change'] = df['post_attrition_rate'] - df['pre_attrition_rate']

hte_results = []
for dept in df['department'].unique():
    dept_data = df[df['department'] == dept]
    treated_change = dept_data[dept_data['treatment'] == 1]['attrition_change'].mean()
    control_change = dept_data[dept_data['treatment'] == 0]['attrition_change'].mean()
    did = treated_change - control_change
    n_treated = dept_data['treatment'].sum()
    n_control = (1 - dept_data['treatment']).sum()
    hte_results.append({
        'Department': dept, 'DiD Effect': did,
        'N Treated': int(n_treated), 'N Control': int(n_control)
    })

hte_df = pd.DataFrame(hte_results).sort_values('DiD Effect')

print("=" * 60)
print("HETEROGENEOUS TREATMENT EFFECTS BY DEPARTMENT")
print("=" * 60)
print(hte_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
colors = [COLORS['treated'] if x < 0 else COLORS['control'] for x in hte_df['DiD Effect']]
bars = ax.barh(hte_df['Department'], hte_df['DiD Effect'], color=colors, edgecolor='white')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.axvline(x=TRUE_ATE, color=COLORS['neutral'], linestyle='--', alpha=0.5, label=f'True ATE ({TRUE_ATE:.1%})')

for bar, val in zip(bars, hte_df['DiD Effect']):
    offset = -0.003 if val < 0 else 0.001
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            f'{val:.1%}', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('DiD Treatment Effect on Attrition Rate')
ax.set_title('Training Program Impact Varies by Department', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/hte_by_department.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretation: Heterogeneous Effects

The training program does not have the same impact everywhere. Support and Sales departments show larger effects than Engineering and Operations. This pattern makes intuitive sense: departments with higher baseline attrition (Support at 22%, Sales at 18%) have more room for improvement, and manager quality may be a more binding constraint in those functions.

**Practical implication:** If L&D budget is limited, the next training cohort should prioritize Support and Sales managers, where the expected ROI per manager trained is 1.2 to 1.5x higher than the average. This is a direct, evidence-based input to the budget allocation decision.

## 5. Sensitivity Analysis

The central worry with any observational causal estimate is: **what if there is an unmeasured confounder?** Maybe managers who were trained were also more motivated, and that motivation (not the training) drove the improvement.

The **Rosenbaum bounds** test asks: how strong would a hidden confounder need to be to explain away our result? The parameter gamma represents the factor by which a hidden confounder could increase the odds of treatment. At gamma = 1, there is no hidden bias. As gamma increases, we allow for stronger unmeasured confounding.

In [ ]:
def rosenbaum_bounds(att, se_val, gammas=np.arange(1.0, 3.1, 0.1)):
    """Simplified Rosenbaum bounds sensitivity analysis."""
    results = []
    z_stat = abs(att / se_val)
    for gamma in gammas:
        adjusted_z = z_stat - np.log(gamma)
        p_upper = 1 - stats.norm.cdf(adjusted_z)
        results.append({'gamma': gamma, 'adjusted_z': adjusted_z, 'p_upper': p_upper})
    return pd.DataFrame(results)

sensitivity = rosenbaum_bounds(abs(att_psm), se)

print("=" * 60)
print("ROSENBAUM SENSITIVITY ANALYSIS")
print("=" * 60)
print("Gamma = how much more likely a hidden confounder makes")
print("        treatment assignment (1.0 = no hidden bias)")
print(f"\n{'Gamma':>8} {'Adjusted z':>12} {'p-value upper':>15}")
print("-" * 40)

for _, row in sensitivity.iterrows():
    flag = '  <-- no longer significant' if row['p_upper'] > 0.05 else ''
    print(f"{row['gamma']:>8.1f} {row['adjusted_z']:>12.3f} {row['p_upper']:>15.4f}{flag}")

critical_gamma = sensitivity.loc[sensitivity['p_upper'] > 0.05, 'gamma'].min()
print(f"\nCritical Gamma: {critical_gamma:.1f}")
print(f"An unmeasured confounder would need to make treatment")
print(f"{critical_gamma:.1f}x more likely to explain away the observed effect.")

### Interpretation: Sensitivity Analysis

The critical gamma tells us the "breaking point" of our causal claim. If the critical gamma is, say, 2.0, it means an unmeasured confounder would need to double the odds of being selected for training to fully explain away our result. That is a fairly strong confounder, given that we have already controlled for experience, rating, team size, department, and pre-treatment attrition.

Higher critical gamma = more robust finding. In practice, a critical gamma above 1.5 is generally considered reasonable for HR studies where the key confounders are observable. If the critical gamma were close to 1.0, the result would be fragile and not trustworthy.

This analysis does not prove there is no unmeasured confounding. It quantifies how large the confounding would need to be, which allows stakeholders to make informed judgments about the plausibility of alternative explanations.

## 6. Method Comparison & Business Recommendations

In [ ]:
summary = pd.DataFrame({
    'Method': ['Naive Comparison', 'Propensity Score Matching', 'Difference-in-Differences', 'True Effect'],
    'Estimate': [naive, att_psm, did_effect, TRUE_ATE],
    'Bias': [naive - TRUE_ATE, att_psm - TRUE_ATE, did_effect - TRUE_ATE, 0.0],
    'Valid?': ['Biased (no adjustment)', 'Causal (CIA assumption)', 'Causal (parallel trends)', 'Ground truth']
})
summary['Estimate (pp)'] = (summary['Estimate'] * 100).round(2).astype(str) + ' pp'

print("=" * 80)
print("METHOD COMPARISON")
print("=" * 80)
print(summary[['Method', 'Estimate (pp)', 'Bias', 'Valid?']].to_string(index=False))
print(f"\nKey finding: Naive comparison overstates the effect by ~{abs(naive)/abs(TRUE_ATE):.1f}x")
print(f"Both causal methods converge on a similar estimate, increasing confidence.")

### Business Implications & ROI

The purpose of this analysis is not the estimate itself. It is the **decision it enables.** Here is how to translate the causal finding into a budget recommendation for HR leadership.

#### ROI Calculation

| Parameter | Value | Source |
|---|---|---|
| Causal effect on attrition | ~4 percentage points | PSM and DiD estimates |
| Average team size | 12 employees | Data |
| Attrition events prevented per manager trained | 12 x 0.04 = 0.48 | Derived |
| Average replacement cost | $120,000 (1.5x avg salary) | Industry benchmark |
| **Savings per manager trained** | **$57,600/year** | 0.48 x $120K |

#### Targeting Recommendations

| Department | Estimated Effect | Expected Savings per Manager | Priority |
|---|---|---|---|
| Support | ~6 pp (1.5x average) | ~$86,400/year | Highest |
| Sales | ~5 pp (1.2x average) | ~$72,000/year | High |
| Engineering | ~4 pp (base) | ~$57,600/year | Standard |
| Operations | ~4 pp (base) | ~$57,600/year | Standard |

#### Key Recommendations for HR Leadership

1. **Continue and expand the training program.** The causal evidence supports a real, meaningful effect of approximately 4 percentage points on team attrition.

2. **Prioritize Support and Sales for the next cohort.** The heterogeneous treatment effect analysis shows these departments benefit most, delivering 1.2 to 1.5x the ROI of other functions.

3. **Do not rely on naive before/after comparisons for future evaluations.** The naive estimate overstated the effect by roughly 2x. Every program evaluation should use causal methods to produce trustworthy estimates.

4. **Consider an RCT for the next rollout.** If budget allows, randomly assigning the next cohort would eliminate the need for observational adjustments entirely and produce the most defensible evidence.

---

## 7. Limitations & Future Work

### Data Limitations

- **Simulated data:** The dataset is synthetically generated with a known treatment effect. Real HR data would contain more noise, more confounders, and the true effect would be unknown. The methods demonstrated here are valid for real data, but the clean recovery of the true effect reflects the simulation design.
- **Single pre/post period:** With only one pre-treatment period, we cannot formally test the parallel trends assumption for DiD. Real implementations should use multiple pre-periods when available.
- **Manager-level aggregation:** Attrition is measured at the team level. Individual employee-level data would allow for richer modeling, including employee fixed effects.

### Methodological Limitations

- **Unmeasured confounders:** PSM can only adjust for observed covariates. If unobserved factors (manager motivation, team culture, organizational politics) drive both selection and outcomes, the estimate will be biased. The sensitivity analysis quantifies this risk but cannot eliminate it.
- **SUTVA assumption:** Both methods assume that one manager's treatment does not affect other managers' outcomes (Stable Unit Treatment Value Assumption). If trained managers share techniques with untrained managers, the control group outcomes would be contaminated, diluting the estimated effect.
- **Nearest-neighbor matching limitations:** 1:1 matching without replacement discards information from unmatched control units. More sophisticated matching methods (kernel matching, genetic matching, CEM) could improve efficiency.
- **Caliper choice:** The 0.05 caliper is a common default but not data-driven. Varying the caliper would be a useful robustness check.

### Future Extensions

- **Doubly robust estimation:** Combine propensity score weighting with outcome modeling (Augmented IPW) for protection against misspecification of either the propensity or outcome model.
- **Double/Debiased Machine Learning:** Use flexible ML models for both the propensity score and the outcome equation, with cross-fitting to avoid overfitting bias.
- **Synthetic Control Method:** For evaluating interventions in small groups (e.g., a single department), construct a weighted combination of control units as the counterfactual.
- **Causal forests (GRF):** For discovering heterogeneous treatment effects without pre-specifying which subgroups to examine.
- **Cost-effectiveness analysis:** Incorporate training program costs to produce a net ROI rather than gross savings estimates.

---

*Built by [Sunidhi Sharma](https://www.linkedin.com/in/sunidhisharma28/) | Senior Data Scientist specializing in People Analytics, Causal Inference, and Responsible AI in HR. 5+ years across Publicis Sapient, Korn Ferry, Infosys, and TCS. MS in Business Analytics, University of Cincinnati.*